# 📚 Notebook 4 — Weaviate with Research Papers

> **Goal:** see Weaviate's **typed-schema + hybrid-search** model.

## 🎯 Learning Goals
- Define a Weaviate **class** with typed properties
- Insert objects with our own pre-computed vectors
- Run **hybrid search** (BM25 keyword + vector)
- Filter with `where` clauses on metadata

## 🍱 Analogy
**Weaviate = a graph database with vectors built-in.**
- Every object belongs to a `class` (like a SQL table) with typed properties.
- Hybrid search blends keyword (BM25) and semantic (vectors) — the query "deep learning for protein folding" finds papers that match by **terms** AND by **meaning**.

> ℹ️ This notebook uses **Weaviate Embedded** which spawns a local instance from Python — no Docker.

In [ ]:
# 📦 Install
# %pip install -q "weaviate-client>=4.5" sentence-transformers pandas

In [ ]:
from IPython.display import SVG
SVG("""<svg viewBox="0 0 720 230" xmlns="http://www.w3.org/2000/svg" width="720"><style>text{font-family:Inter,system-ui,sans-serif}</style><rect width="720" height="230" fill="#f8fafc" rx="12"/><rect x="20" y="60" width="140" height="80" rx="10" fill="#dbeafe" stroke="#2563eb"/><text x="90" y="90" text-anchor="middle" font-size="13" font-weight="700" fill="#1e3a8a">Papers</text><text x="90" y="110" text-anchor="middle" font-size="11" fill="#1e3a8a">15 abstracts</text><text x="90" y="125" text-anchor="middle" font-size="11" fill="#1e3a8a">+ field, year</text><rect x="200" y="60" width="140" height="80" rx="10" fill="#ede9fe" stroke="#7c3aed"/><text x="270" y="90" text-anchor="middle" font-size="13" font-weight="700" fill="#5b21b6">Vectorizer</text><text x="270" y="110" text-anchor="middle" font-size="11" fill="#5b21b6">MiniLM via client</text><text x="270" y="125" text-anchor="middle" font-size="11" fill="#5b21b6">→ vectors</text><rect x="380" y="60" width="140" height="80" rx="10" fill="#fef3c7" stroke="#f59e0b"/><text x="450" y="90" text-anchor="middle" font-size="13" font-weight="700" fill="#78350f">Class schema</text><text x="450" y="110" text-anchor="middle" font-size="11" fill="#78350f">typed properties</text><text x="450" y="125" text-anchor="middle" font-size="11" fill="#78350f">+ vector index</text><rect x="560" y="60" width="140" height="80" rx="10" fill="#dcfce7" stroke="#15803d"/><text x="630" y="90" text-anchor="middle" font-size="13" font-weight="700" fill="#065f46">GraphQL-style</text><text x="630" y="110" text-anchor="middle" font-size="11" fill="#065f46">nearText + where</text><text x="630" y="125" text-anchor="middle" font-size="11" fill="#065f46">hybrid search</text><text x="172" y="105" font-size="22" fill="#475569">→</text><text x="352" y="105" font-size="22" fill="#475569">→</text><text x="532" y="105" font-size="22" fill="#475569">→</text><text x="360" y="35" text-anchor="middle" font-size="14" font-weight="700" fill="#0f172a">Weaviate pipeline</text><text x="360" y="180" text-anchor="middle" font-size="11" fill="#475569">Weaviate uses a typed schema (classes + properties) like a graph database</text></svg>""")

In [ ]:
# 📚 Research paper abstracts across CS sub-fields
papers = [
    {"title": "Attention Is All You Need",                       "field": "NLP",    "year": 2017, "abstract": "We propose the Transformer, a network architecture based solely on attention mechanisms, dispensing with recurrence and convolutions for sequence modeling."},
    {"title": "BERT: Pre-training of Deep Bidirectional Transformers", "field": "NLP", "year": 2018, "abstract": "BERT is a new method of pre-training language representations that achieves state-of-the-art results across many natural language tasks."},
    {"title": "GPT-3: Language Models are Few-Shot Learners",   "field": "NLP",    "year": 2020, "abstract": "We train GPT-3, an autoregressive language model with 175 billion parameters, and study its few-shot in-context learning abilities."},
    {"title": "ResNet: Deep Residual Learning for Image Recognition", "field": "CV", "year": 2015, "abstract": "We present a residual learning framework that eases training of substantially deeper neural networks for image recognition."},
    {"title": "Vision Transformer",                              "field": "CV",     "year": 2020, "abstract": "An image is worth 16x16 words: pure transformer architectures applied directly to image patches achieve excellent results on classification."},
    {"title": "YOLO: Real-Time Object Detection",                "field": "CV",     "year": 2016, "abstract": "You Only Look Once frames object detection as a single regression problem, enabling unified real-time detection at high frame rates."},
    {"title": "AlphaFold: Protein Structure Prediction",         "field": "BioAI",  "year": 2021, "abstract": "We describe a deep learning system that predicts protein 3D structures with accuracy competitive with experimental methods."},
    {"title": "DeepMind AlphaGo",                                "field": "RL",     "year": 2016, "abstract": "We introduce a Go-playing program that combines deep neural networks and Monte Carlo tree search to defeat the world champion."},
    {"title": "Proximal Policy Optimization",                    "field": "RL",     "year": 2017, "abstract": "We propose a family of policy gradient methods for reinforcement learning that alternate sampling and optimizing a clipped surrogate objective."},
    {"title": "Generative Adversarial Networks",                 "field": "GenAI",  "year": 2014, "abstract": "We propose a framework for estimating generative models via an adversarial process pitting a generator against a discriminator network."},
    {"title": "Diffusion Models for Image Synthesis",            "field": "GenAI",  "year": 2020, "abstract": "Denoising diffusion probabilistic models achieve high-quality image synthesis by reversing a gradual noising process."},
    {"title": "Stable Diffusion: Latent Diffusion Models",       "field": "GenAI",  "year": 2022, "abstract": "Latent diffusion models perform diffusion in a compressed latent space enabling high-resolution image synthesis on consumer GPUs."},
    {"title": "Dense Passage Retrieval for QA",                  "field": "IR",     "year": 2020, "abstract": "We show that dense vector representations trained with a small number of questions and passages outperform traditional lexical retrieval methods."},
    {"title": "ColBERT: Late Interaction over BERT",             "field": "IR",     "year": 2020, "abstract": "ColBERT enables fine-grained late interaction between query and document terms while remaining efficient enough for large-scale retrieval."},
    {"title": "FAISS: Billion-scale Similarity Search",          "field": "IR",     "year": 2017, "abstract": "We describe FAISS, a library for efficient similarity search of dense vectors that scales to billions of items on a single machine."},
]
import pandas as pd
pd.DataFrame(papers).head()

In [ ]:
# 🚀 Start an embedded Weaviate (no Docker, no separate server)
import weaviate
from weaviate.embedded import EmbeddedOptions
import weaviate.classes.config as wc
from weaviate.classes.query import Filter, MetadataQuery

client = weaviate.connect_to_embedded(version="1.24.10")
print("Weaviate ready:", client.is_ready())

In [ ]:
# 📐 Define a Paper class — typed schema (like a SQL table)
if client.collections.exists("Paper"):
    client.collections.delete("Paper")

client.collections.create(
    name="Paper",
    properties=[
        wc.Property(name="title",    data_type=wc.DataType.TEXT),
        wc.Property(name="field",    data_type=wc.DataType.TEXT),
        wc.Property(name="year",     data_type=wc.DataType.INT),
        wc.Property(name="abstract", data_type=wc.DataType.TEXT),
    ],
    vectorizer_config=wc.Configure.Vectorizer.none(),   # we supply our own vectors
)
print("Class 'Paper' created")

In [ ]:
# 🧠 Encode abstracts with MiniLM, then bulk-insert
from sentence_transformers import SentenceTransformer
encoder = SentenceTransformer("all-MiniLM-L6-v2")
abstracts = [p["abstract"] for p in papers]
vectors = encoder.encode(abstracts).tolist()

paper_col = client.collections.get("Paper")
with paper_col.batch.dynamic() as batch:
    for p, v in zip(papers, vectors):
        batch.add_object(properties=p, vector=v)
print("Inserted", len(papers), "papers")

In [ ]:
# 🔍 Vector search using a query vector we compute ourselves
def show(res, label):
    print(f"\n🔎 {label}")
    rows = []
    for o in res.objects:
        rows.append({"title": o.properties["title"], "field": o.properties["field"],
                     "year": o.properties["year"], "distance": round(o.metadata.distance, 3) if o.metadata.distance else None})
    return pd.DataFrame(rows)

q = "transformers and attention for language modeling"
qv = encoder.encode([q]).tolist()[0]
res = paper_col.query.near_vector(near_vector=qv, limit=5, return_metadata=MetadataQuery(distance=True))
show(res, q)

In [ ]:
# 🎯 Filtered search — only papers from 2020 onward
q = "deep learning for biological problems"
qv = encoder.encode([q]).tolist()[0]
res = paper_col.query.near_vector(
    near_vector=qv,
    limit=5,
    filters=Filter.by_property("year").greater_or_equal(2020),
    return_metadata=MetadataQuery(distance=True),
)
show(res, q + " [year >= 2020]")

In [ ]:
# 🌗 Hybrid search — combine BM25 keyword + vector similarity (alpha=0.5 is balanced)
res = paper_col.query.hybrid(
    query="diffusion image generation",
    vector=encoder.encode(["diffusion image generation"]).tolist()[0],
    alpha=0.5,
    limit=5,
    return_metadata=MetadataQuery(score=True),
)
print("\n🌗 Hybrid: 'diffusion image generation'")
pd.DataFrame([{"title": o.properties["title"], "field": o.properties["field"],
               "score": round(o.metadata.score, 3)} for o in res.objects])

In [ ]:
# 🧹 Always close the embedded instance
client.close()

## 🏋️ Exercises
1. Add a `citations` int property and re-create the schema. Filter for `citations > 1000`.
2. Try `alpha=0.0` (pure keyword) and `alpha=1.0` (pure vector) — observe how rankings shift.
3. Replace `vectorizer_config=...none()` with `text2vec-transformers` (in a Docker setup) so Weaviate auto-embeds.

## 🎓 Recap — When to use Weaviate
✅ Want hybrid (keyword + vector) search out of the box
✅ Schema-driven, multi-tenant, GraphQL API
❌ Pure simple FAISS-like need → too heavy